# SHACL 入門

- [W3C | Shapes Constraint Language (SHACL), §1.4 SHACL Example](https://www.w3.org/TR/shacl/#shacl-example)
- [RDFLib/pySHACL: A Python validator for SHACL](https://github.com/RDFLib/pySHACL)

In [24]:
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, SH
from rdflib.extras.external_graph_libs import rdflib_to_networkx_multidigraph
from yfiles_jupyter_graphs import GraphWidget, Layout
from IPython.display import display

EX = Namespace("http://example.com/ns#")


In [25]:
EX

Namespace('http://example.com/ns#')

In [26]:
def show_rdf_graph(rdf_graph: Graph): # -> GraphWidget:
    """Visualize an RDFLib graph with yFiles."""
    nx_graph = rdflib_to_networkx_multidigraph(
        rdf_graph,
        edge_attrs=lambda s, p, o: {
            "key": p,
            "label": p.n3(rdf_graph.namespace_manager),
        },
    )
    widget = GraphWidget(
        graph=nx_graph,
        graph_layout=Layout.HIERARCHICAL,
    )
    display(widget)
    # return widget

## data graph (検査対象)

In [27]:
data_ttl: str = """
@prefix ex: <http://example.com/ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:Alice
    a ex:Person ;
    ex:ssn "987-65-432A" .

ex:Bob
    a ex:Person ;
    ex:ssn "123-45-6789" ;
    ex:ssn "124-35-6789" .

ex:Calvin
    a ex:Person ;
    ex:birthDate "1971-07-07"^^xsd:date ;
    ex:worksFor ex:UntypedCompany .
"""

data_graph = Graph().parse(data=data_ttl, format="turtle")
data_graph.bind("ex", EX)

print(f"Triple count: {len(data_graph)}")

for person in (EX.Alice, EX.Bob, EX.Calvin):
    print(f"\n{person.n3(data_graph.namespace_manager)}")
    for predicate, obj in data_graph.predicate_objects(person):
        print(f"  {predicate.n3(data_graph.namespace_manager)} → {obj}")

Triple count: 8

ex:Alice
  rdf:type → http://example.com/ns#Person
  ex:ssn → 987-65-432A

ex:Bob
  rdf:type → http://example.com/ns#Person
  ex:ssn → 123-45-6789
  ex:ssn → 124-35-6789

ex:Calvin
  rdf:type → http://example.com/ns#Person
  ex:birthDate → 1971-07-07
  ex:worksFor → http://example.com/ns#UntypedCompany


In [28]:
show_rdf_graph(data_graph)

GraphWidget(layout=Layout(height='500px', width='100%'))

## shape graph (スキーマ)

In [29]:
shapes_ttl: str = r"""
@prefix ex: <http://example.com/ns#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape
    a sh:NodeShape ;
    sh:targetClass ex:Person ;

    sh:property [
        sh:path ex:ssn ;
        sh:maxCount 1 ;
        sh:datatype xsd:string ;
        sh:pattern "^\\d{3}-\\d{2}-\\d{4}$" ;
    ] ;

    sh:property [
        sh:path ex:worksFor ;
        sh:class ex:Company ;
        sh:nodeKind sh:IRI ;
    ] ;

    sh:closed true ;
    sh:ignoredProperties ( rdf:type ) .
"""

shapes_graph = Graph().parse(
    data=shapes_ttl,
    format="turtle",
)

shapes_graph.bind("ex", EX)

print("Shape triples:", len(shapes_graph))


Shape triples: 15


In [30]:
show_rdf_graph(shapes_graph)


GraphWidget(layout=Layout(height='660px', width='100%'))

In [31]:
person_shape = EX.PersonShape

print(
    "Target:",
    shapes_graph.value(person_shape, SH.targetClass),
)

for property_shape in shapes_graph.objects(person_shape, SH.property):
    path = shapes_graph.value(property_shape, SH.path)

    print(f"\nProperty: {path}")

    for p, o in shapes_graph.predicate_objects(property_shape):
        print(" ", p.n3(shapes_graph.namespace_manager), "→", o)


Target: http://example.com/ns#Person

Property: http://example.com/ns#ssn
  sh:path → http://example.com/ns#ssn
  sh:maxCount → 1
  sh:datatype → http://www.w3.org/2001/XMLSchema#string
  sh:pattern → ^\d{3}-\d{2}-\d{4}$

Property: http://example.com/ns#worksFor
  sh:path → http://example.com/ns#worksFor
  sh:class → http://example.com/ns#Company
  sh:nodeKind → http://www.w3.org/ns/shacl#IRI


## Validation 実行

In [33]:
from pyshacl import validate

conforms, results_graph, results_text = validate(
    data_graph,
    shacl_graph=shapes_graph,
    inference="none",
    abort_on_first=False,
)

print("Conforms:", conforms)

Conforms: False


In [34]:
print(results_text)

Validation Report
Conforms: False
Results (4):
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class ex:Company ; sh:nodeKind sh:IRI ; sh:path ex:worksFor ]
	Focus Node: ex:Calvin
	Value Node: ex:UntypedCompany
	Result Path: ex:worksFor
	Message: Value does not have class ex:Company
Constraint Violation in ClosedConstraintComponent (http://www.w3.org/ns/shacl#ClosedConstraintComponent):
	Severity: sh:Violation
	Source Shape: ex:PersonShape
	Focus Node: ex:Calvin
	Value Node: Literal("1971-07-07", datatype=xsd:date)
	Result Path: ex:birthDate
	Message: Node ex:Calvin is closed. It cannot have value: Literal("1971-07-07", datatype=xsd:date)
Constraint Violation in MaxCountConstraintComponent (http://www.w3.org/ns/shacl#MaxCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:path ex:ssn ; sh:pattern 

In [35]:
show_rdf_graph(results_graph)

GraphWidget(layout=Layout(height='800px', width='100%'))

In [36]:
validation_results = list(
    results_graph.subjects(
        RDF.type,
        SH.ValidationResult,
    )
)

print("Violation count:", len(validation_results))

for result in validation_results:
    focus = results_graph.value(result, SH.focusNode)
    path = results_graph.value(result, SH.resultPath)
    component = results_graph.value(result, SH.sourceConstraintComponent)

    print(
        str(focus).split("#")[-1],
        "|",
        str(path).split("#")[-1],
        "|",
        str(component).split("#")[-1],
    )


Violation count: 4
Bob | ssn | MaxCountConstraintComponent
Alice | ssn | PatternConstraintComponent
Calvin | worksFor | ClassConstraintComponent
Calvin | birthDate | ClosedConstraintComponent


## データを修正

In [37]:
from rdflib import Literal

fixed_graph = Graph()
fixed_graph += data_graph

# Alice: 不正な SSN を修正
fixed_graph.set(
    (
        EX.Alice,
        EX.ssn,
        Literal("987-65-4321"),
    )
)

# Bob: 余分な SSN を削除
fixed_graph.remove(
    (
        EX.Bob,
        EX.ssn,
        Literal("124-35-6789"),
    )
)

# Calvin: 勤務先が Company であると記述
fixed_graph.add(
    (
        EX.UntypedCompany,
        RDF.type,
        EX.Company,
    )
)

# Calvin: 今回の Shape で許可されない属性を削除
fixed_graph.remove(
    (
        EX.Calvin,
        EX.birthDate,
        None,
    )
)

print("Fixed graph triples:", len(fixed_graph))

Fixed graph triples: 7


In [38]:
fixed_conforms, fixed_report, fixed_text = validate(
    fixed_graph,
    shacl_graph=shapes_graph,
    inference="none",
)

print("Conforms:", fixed_conforms)

print(
    "Validation results:",
    len(
        list(
            fixed_report.subjects(
                RDF.type,
                SH.ValidationResult,
            )
        )
    ),
)

Conforms: True
Validation results: 0


In [39]:
show_rdf_graph(fixed_report)

GraphWidget(layout=Layout(height='500px', width='100%'))